# KOveri Primer Designer

Interactive workflow for designing CRISPR KO genotyping primers around user-provided gRNAs.

In [1]:
from pathlib import Path
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from config import KOVERI_CONFIG, DEFAULT_OUTPUT_DIR
from utils.designer import design_koveri_primers

OUTPUT_DIR = DEFAULT_OUTPUT_DIR
INPUT_FASTA = OUTPUT_DIR / 'Tsix_gRNAs.fa'

config = dict(KOVERI_CONFIG)
config['min_informative_snps'] = 3
config['flank_steps'] = [250, 500, 750, 1000, 1500, 2500, 5000]
config['max_amplicon_size'] = 5000
config['blast_specificity'] = True
config['top_n_output'] = 20
config['specificity_candidates'] = 60

INPUT_FASTA, OUTPUT_DIR

(PosixPath('/Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix/Tsix_gRNAs.fa'),
 PosixPath('/Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix'))

Tune the values above when a region is difficult. The most common knobs are `min_informative_snps`, `flank_steps`, `max_amplicon_size`, and `blast_specificity`.

Top candidates are selected from the strictest passing post-Primer3 filter tier only. If one strict pair passes, the output is not padded with relaxed pairs.

In [2]:
with Progress(
    SpinnerColumn(),
    TextColumn('[progress.description]{task.description}'),
    BarColumn(),
    TimeElapsedColumn(),
    transient=False,
) as progress:
    task = progress.add_task('Running KOveri primer design...', total=None)
    result = design_koveri_primers(INPUT_FASTA, OUTPUT_DIR, config)
    progress.update(task, description='KOveri primer design complete')
target = result['target']
print(f'Target interval: {target.chrom}:{target.start}-{target.end}')
print(f'Candidate pairs: {len(result["candidate_rows"])}')
print(f'Output directory: {result["output_dir"]}')
print(f'HTML report: {result["output_dir"] / "KOveri_report.html"}')

Output()

Target interval: chrX:103447474-103448345
Candidate pairs: 354
Output directory: /Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix
HTML report: /Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix/KOveri_report.html


In [3]:
import pandas as pd

guide_hits = pd.read_csv(OUTPUT_DIR / 'KOveri_guide_hits.csv')
guide_hits

,Guide_ID,Guide_Seq,Chromosome,Guide_Start,Guide_End,Guide_Strand,Exact_Hit_Count_For_Guide,Partial_Hit_Count_For_Guide,Selected_For_Target
0,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103449155,103449174,-,9,0,False
1,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448647,103448666,-,9,0,False
2,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448580,103448599,-,9,0,False
3,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448349,103448368,+,9,0,False
4,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448884,103448903,-,9,0,False
5,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103449087,103449106,-,9,0,False
6,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448613,103448632,-,9,0,False
7,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448430,103448449,+,9,0,False
8,TsixMP,CTATCGGTTCGGGACTACGC,chrX,103448326,103448345,-,9,0,True
9,TMP,ACTGAATAAACACTACCGGG,chrX,103447474,103447493,+,2,0,True


In [4]:
top = pd.read_csv(OUTPUT_DIR / 'KOveri_primer_top_candidates.csv')
display_cols = [
    'Chromosome', 'Amplicon_Start', 'Amplicon_End', 'Amplicon_Size',
    'SNP_Count_In_Amplicon', 'Left_Primer_Seq', 'Right_Primer_Seq',
    'Primer_Overlaps_SNP', 'Primer_Pair_Specificity_Pass'
]
top[display_cols]

,Chromosome,Amplicon_Start,Amplicon_End,Amplicon_Size,SNP_Count_In_Amplicon,Left_Primer_Seq,Right_Primer_Seq,Primer_Overlaps_SNP,Primer_Pair_Specificity_Pass
0,chrX,103447425,103448802,1378,13,ATGAGACTGTGTTGGAGCTGAG,GTGATAGCCCAGATCCTCGAC,False,False
1,chrX,103447425,103448803,1379,13,ATGAGACTGTGTTGGAGCTGAG,AGTGATAGCCCAGATCCTCGAC,False,False
2,chrX,103447425,103448804,1380,13,ATGAGACTGTGTTGGAGCTGAG,TAGTGATAGCCCAGATCCTCGAC,False,False
3,chrX,103447425,103448805,1381,13,ATGAGACTGTGTTGGAGCTGAG,TTAGTGATAGCCCAGATCCTCG,False,False
4,chrX,103447425,103448806,1382,13,ATGAGACTGTGTTGGAGCTGAG,TTTAGTGATAGCCCAGATCCTCG,False,False
5,chrX,103447323,103448802,1480,13,ACCCAGTCTTTGAGACCGTAAG,GTGATAGCCCAGATCCTCGAC,False,False
6,chrX,103447323,103448803,1481,13,ACCCAGTCTTTGAGACCGTAAG,AGTGATAGCCCAGATCCTCGAC,False,False
7,chrX,103447323,103448805,1483,13,ACCCAGTCTTTGAGACCGTAAG,TTAGTGATAGCCCAGATCCTCG,False,False
8,chrX,103447319,103448802,1484,13,TCAAACCCAGTCTTTGAGACCG,GTGATAGCCCAGATCCTCGAC,False,False
9,chrX,103447319,103448803,1485,13,TCAAACCCAGTCTTTGAGACCG,AGTGATAGCCCAGATCCTCGAC,False,False


In [5]:
from IPython.display import IFrame, display

display(IFrame(src=str(OUTPUT_DIR / 'KOveri_report.html'), width='100%', height=700))

In [6]:
print((OUTPUT_DIR / 'KOveri_summary.txt').read_text())

KOveri primer design summary

Input FASTA: /Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix/Tsix_gRNAs.fa
Output directory: /Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/AmpliconSeq-KOveri-Tsix
Guides: 10
Exact guide hits: 11
Selected protected edit interval: chrX:103447474-103448345
Minimum informative SNP goal: 3
Candidate primer pairs: 354
Top primer pairs reported: 20

Post-Primer3 filters
--------------------
Selected filter tier: relaxed_specificity
Tier counts: {'strict_all': 0, 'relaxed_amplicon': 0, 'relaxed_specificity': 33, 'relaxed_gc_tm': 0, 'failed': 321}
Hard-filter passing candidates: 33
Hard-filter failing candidates: 321
Tier explanations:
  - strict_all: All hard filters and all soft preference filters passed.
  - relaxed_amplicon: Hard filters passed with strict specificity; only the 400-700 bp amplicon preference was relaxed.
  - rela